# TV2.2b: EDA dữ liệu silver (10 trạm chuẩn, 2023–2025)

Notebook chỉ gọi hàm trong `batch/eda_tv2.py` và `batch/data_report_tv2.py`; mọi phép tổng hợp chạy bằng **Spark** trên data
lake (`WQ_LAKE_PATH`, cục bộ hoặc HDFS), chỉ kết quả nhỏ về pandas để vẽ. Hình được lưu vào `docs/images/tv2_*.png`,
bảng vào `docs/report/tables/tv2_*.csv`. Chạy lại toàn bộ không cần notebook: `python -m batch.data_report_tv2` rồi
`python -m batch.eda_tv2`.

Chạy Jupyter trong container (xem `notebooks/README.md`), thư mục làm việc là gốc repo.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from IPython.display import Image, display

from batch import data_report_tv2 as rep, eda_tv2 as eda
from wq.config import get_settings
from wq.spark_utils import build_spark

TABLES, IMAGES = Path("docs/report/tables"), Path("docs/images")
spark = build_spark("tv2-eda-notebook", master="local[*]")
meta = rep.load_stations("config/stations.json")
silver = rep.read_silver_spark(spark, get_settings().lake_path, list(meta["station_id"])).cache()
print(f"{silver.count():,} dòng silver")

## 1. Kiểm chứng silver và tỉ lệ thiếu (TV2.1)

In [ ]:
checks = rep.silver_checks(silver, "2023-01-01", "2026-01-01")
display(checks)
miss = rep.missingness_by_split(silver).toPandas()
display(miss[miss.split == "all"].pivot(index="station_id", columns="variable", values="missing_rate").round(3))

In [ ]:
display(Image(IMAGES / "tv2_missingness.png"))

## 2. Phân bố độ dài đoạn thiếu (gửi TV3)

`tv2_gap_lengths.csv` (`station_id, variable, gap_steps, count`) là đầu vào để TV3.1 lấy mẫu độ dài khối che theo phân bố
thật. Biến mà trạm không đo (100% thiếu) đã bị loại.

In [ ]:
runs = rep.gap_runs(silver)
counts = rep.gap_length_counts(runs).toPandas()
display(rep.bin_gaps(counts).pivot(index="bin", columns="variable", values="share_missing_steps")
        .reindex([b[2] for b in rep.GAP_BINS]).round(3))
display(rep.gap_summary(runs).toPandas().sort_values("max_gap_steps", ascending=False).head(10))

In [ ]:
display(Image(IMAGES / "tv2_gap_lengths.png"))

## 3. Phân bố từng biến

In [ ]:
stats = eda.variable_stats(silver).toPandas()
display(stats[stats.variable == "dissolved_oxygen"].round(2))
eda.plot_distributions(stats, meta, IMAGES / "tv2_distributions.png")
display(Image(IMAGES / "tv2_distributions.png"))

## 4. Chu kỳ ngày của DO và nhiệt độ (giờ mặt trời địa phương)

In [ ]:
diurnal = eda.diurnal_profile(silver, meta).toPandas()
amp = (diurnal.groupby(["station_id", "variable", "season"])["anomaly"].agg(lambda s: s.max() - s.min())
       .unstack("season").round(2))
display(amp)   # biên độ dao động ngày (max - min của trung bình theo giờ)
eda.plot_diurnal(diurnal, meta, IMAGES / "tv2_diurnal.png")
display(Image(IMAGES / "tv2_diurnal.png"))

## 5. Theo mùa

In [ ]:
monthly = eda.monthly_profile(silver).toPandas()
eda.plot_seasonal(monthly, meta, IMAGES / "tv2_seasonal.png")
display(Image(IMAGES / "tv2_seasonal.png"))

## 6. Tương quan giữa các biến (tính trong từng trạm)

In [ ]:
corr = eda.within_station_corr(silver)
mat = eda.plot_correlation(corr, IMAGES / "tv2_correlation.png")
display(mat.round(2))
display(Image(IMAGES / "tv2_correlation.png"))

In [ ]:
silver.unpersist()
spark.stop()